<a href="https://colab.research.google.com/github/iguirote/Contabilidade_Trabalho/blob/main/Contabilidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Calculos

In [ ]:
#Receita Líquida = receita bruta – deduções(Impostos sobre Venda - Devoluções - Descontos Incondicionais - Abatimentos)

In [ ]:
#Situacão Líquida = Ativo − Passivo

In [ ]:
#Margem de contribuição = Preço de venda(Receita bruta) – Custos variáveis – despesas variáveis

In [ ]:
#EBITDA = lucro operacional + depreciação + amortização
# Margem EBITDA (%) =  (EBITDA / receita líquida) x 100

In [3]:
import pandas as pd

def ler_planilha(caminho):
    df = pd.read_excel(
        caminho,
        header=0,
        names=["codigo_erp", "conta contabil", "valor_2024", "valor_2025", "classificacao"],
        usecols=[0, 1, 2, 3, 4]
    )

    for col in ["valor_2024", "valor_2025"]:
        # CORREÇÃO 1: Só faz a limpeza de texto se a coluna realmente for lida como texto (object).
        # Impede que números já formatados corretamente no Excel sejam multiplicados por engano.
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)

        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    return df.to_dict(orient="records")


def classificar_conta(contas):
    balanco = {
        "ativo": [],
        "passivo": [],
        "receita": [],
        "imposto": [],
        "custo": [],
        "despesa": []
    }

    for conta in contas:
        classe = str(conta.get("classificacao", "")).lower()
        nome = str(conta.get("conta contabil", "")).lower()

        if "ativo" in classe:
            balanco["ativo"].append(conta)
        elif "passivo" in classe:
            balanco["passivo"].append(conta)
        elif "receita" in classe or "receita" in nome:
            balanco["receita"].append(conta)
        elif "imposto" in classe or "imposto" in nome:
            balanco["imposto"].append(conta)
        elif "custo" in classe or "custo" in nome or "cmv" in nome:
            balanco["custo"].append(conta)
        # CORREÇÃO 2: Mudamos de 'else' para 'elif' para garantir que apenas
        # despesas reais entrem aqui, ignorando lixo ou linhas em branco.
        elif "despesa" in classe or "despesa" in nome:
            balanco["despesa"].append(conta)

    return balanco


def exibir_ano(ano, balanco):
    chave = f"valor_{ano}"

    # Somatórios das categorias
    ativo = sum(item[chave] for item in balanco["ativo"])
    passivo = sum(item[chave] for item in balanco["passivo"])
    receita = sum(item[chave] for item in balanco["receita"])
    imposto = sum(item[chave] for item in balanco["imposto"])
    custo = sum(item[chave] for item in balanco["custo"])
    despesa = sum(item[chave] for item in balanco["despesa"])

    situacao_liquida = ativo - passivo
    receita_liquida = receita - imposto
    margem_contribuicao = receita_liquida - custo
    ebitda = margem_contribuicao - despesa

    print(f"\n<==========> RELEVANTE {ano} <==========>")
    print(f"Ativo Total            : R$ {ativo:>12,.2f}")
    print(f"Passivo Total          : R$ {passivo:>12,.2f}")
    print(f"Situação Líquida       : R$ {situacao_liquida:>12,.2f}")
    print("------------------------------------------")
    print(f"Receita Líquida        : R$ {receita_liquida:>12,.2f}")
    print(f"Margem de Contribuição : R$ {margem_contribuicao:>12,.2f}")
    print(f"EBITDA                 : R$ {ebitda:>12,.2f}")
    print("<========================================>")


def main():
    caminho = "/content/contas.xlsx"

    try:
        contas = ler_planilha(caminho)
    except FileNotFoundError:
        print("Arquivo não encontrado. Verifique o caminho especificado.")
        return

    balanco = classificar_conta(contas)

    # Exibe apenas o resumo final de cada período operacional
    exibir_ano(2024, balanco)
    exibir_ano(2025, balanco)


if __name__ == "__main__":
    main()


<==========> RELEVANTE 2024 <==========>
Ativo Total            : R$   231,000.00
Passivo Total          : R$   170,500.00
Situação Líquida       : R$    60,500.00
------------------------------------------
Receita Líquida        : R$   130,000.00
Margem de Contribuição : R$    48,000.00
EBITDA                 : R$   -22,800.00
<========================================>

<==========> RELEVANTE 2025 <==========>
Ativo Total            : R$   268,300.00
Passivo Total          : R$   191,700.00
Situação Líquida       : R$    76,600.00
------------------------------------------
Receita Líquida        : R$   -78,000.00
Margem de Contribuição : R$  -186,000.00
EBITDA                 : R$  -276,500.00
<========================================>
